# Assignment 3: Two Averages, One Argument

Calculations using the supplied teaching prices file.

In [ ]:
import pandas as pd

df = pd.read_csv("teaching_prices_clean.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()

## Mean, median and price change

The two city prices are averaged for each item and date.

In [ ]:
weekly = (df.groupby(["date", "item"], as_index=False)["price"]
          .mean().rename(columns={"price": "city_average"}))

results = []

for item, group in weekly.groupby("item"):
    group = group.sort_values("date")
    first = group.iloc[0]["city_average"]
    last = group.iloc[-1]["city_average"]

    results.append({
        "Item": item,
        "Mean": group["city_average"].mean(),
        "Median": group["city_average"].median(),
        "First average": first,
        "Last average": last,
        "Change (%)": (last - first) / first * 100
    })

summary = pd.DataFrame(results)
summary.round(2)

## One item with and without a possible flagged value

The CSV does not contain a flag column. For a sensitivity check, I used the 99.5 Staple A observation for City 1 on 8 January as a possible low value. It is not officially flagged in the supplied file.

In [ ]:
staple = df[df["item"] == "Staple A"].copy()

candidate = staple[
    (staple["date"] == "2026-01-08") &
    (staple["city"] == "City 1") &
    (staple["price"] == 99.5)
]

staple_without = staple.drop(candidate.index)

sensitivity = pd.DataFrame([
    {
        "Version": "With all observations",
        "Count": len(staple),
        "Mean": staple["price"].mean(),
        "Median": staple["price"].median()
    },
    {
        "Version": "Without 99.5 observation",
        "Count": len(staple_without),
        "Mean": staple_without["price"].mean(),
        "Median": staple_without["price"].median()
    }
])

sensitivity.round(2)

## Basic checks

In [ ]:
print("Rows:", len(df))
print("Items:", df["item"].nunique())
print("Cities:", df["city"].nunique())
print("Dates:", df["date"].nunique())
print("Missing values:", df.isna().sum().sum())
print("Data label:", df["data_label"].iloc[0])